# HF Korean Project v2: KLUE-STS 회귀 기반 두 문장 관계 판별

이 노트북은 `HF_korean_project.ipynb`의 두 문장 관계 판별 흐름을 회귀 기반으로 바꾼 버전입니다. KLUE-STS의 `binary-label`을 직접 맞추는 대신 `real-label` 유사도 점수를 먼저 예측하고, 예측 점수에 threshold를 적용해 positive/negative를 판별합니다.

## 라이브러리 의존성

uv 환경에 필요한 라이브러리가 설치되어 있다는 전제로 작성했습니다. 이 노트북에서 사용하는 주요 패키지는 `transformers`, `datasets`, `accelerate`, `torch`, `numpy`, `wandb`입니다. 별도의 `pip install` 코드는 넣지 않았습니다.

# (1) Hugging Face 데이터셋 불러오기

KLUE-STS 데이터셋을 불러옵니다. 이번 버전에서는 `labels` 안의 `real-label`을 회귀 학습 정답으로 사용하고, threshold 평가를 위해 공식 `validation` split을 test처럼 사용합니다.

In [1]:
from datasets import DatasetDict, load_dataset

klue_sts_dataset = load_dataset("klue", "sts")
print(klue_sts_dataset)
print(klue_sts_dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['guid', 'source', 'sentence1', 'sentence2', 'labels'],
        num_rows: 11668
    })
    validation: Dataset({
        features: ['guid', 'source', 'sentence1', 'sentence2', 'labels'],
        num_rows: 519
    })
})
{'guid': 'klue-sts-v1_train_00000', 'source': 'airbnb-rtt', 'sentence1': '숙소 위치는 찾기 쉽고 일반적인 한국의 반지하 숙소입니다.', 'sentence2': '숙박시설의 위치는 쉽게 찾을 수 있고 한국의 대표적인 반지하 숙박시설입니다.', 'labels': {'label': 3.7, 'real-label': 3.714285714285714, 'binary-label': 1}}


데이터셋 컬럼과 샘플 구조를 확인합니다. `sentence1`, `sentence2`는 모델 입력이고, `labels['real-label']`은 0~5 범위의 유사도 점수입니다.

In [2]:
train = klue_sts_dataset["train"]
cols = train.column_names
print(cols)

for i in range(3):
    for col in cols:
        print(col, ":", train[col][i])
    print()

['guid', 'source', 'sentence1', 'sentence2', 'labels']
guid : klue-sts-v1_train_00000
source : airbnb-rtt
sentence1 : 숙소 위치는 찾기 쉽고 일반적인 한국의 반지하 숙소입니다.
sentence2 : 숙박시설의 위치는 쉽게 찾을 수 있고 한국의 대표적인 반지하 숙박시설입니다.
labels : {'label': 3.7, 'real-label': 3.714285714285714, 'binary-label': 1}

guid : klue-sts-v1_train_00001
source : policy-sampled
sentence1 : 위반행위 조사 등을 거부·방해·기피한 자는 500만원 이하 과태료 부과 대상이다.
sentence2 : 시민들 스스로 자발적인 예방 노력을 한 것은 아산 뿐만이 아니었다.
labels : {'label': 0.0, 'real-label': 0.0, 'binary-label': 0}

guid : klue-sts-v1_train_00002
source : paraKQC-sampled
sentence1 : 회사가 보낸 메일은 이 지메일이 아니라 다른 지메일 계정으로 전달해줘.
sentence2 : 사람들이 주로 네이버 메일을 쓰는 이유를 알려줘
labels : {'label': 0.3, 'real-label': 0.3333333333333333, 'binary-label': 0}



# (2) 회귀 라벨 구성 및 전체 train 사용

기존 버전에서는 원본 train의 10%를 evaluation 용도로 분리했지만, 이번 버전에서는 학습 데이터가 많지 않으므로 KLUE-STS train 전체를 학습에 사용합니다. 공식 `validation` split은 최종 평가와 learning rate 선택 기준으로 사용합니다.

In [3]:
def extract_regression_label(example):
    real_label = float(example["labels"]["real-label"])
    official_binary_label = int(example["labels"]["binary-label"])
    return {
        "label": real_label,
        "real_label": real_label,
        "official_binary_label": official_binary_label,
    }


klue_sts_regression = klue_sts_dataset.map(extract_regression_label)

hf_dataset = DatasetDict(
    {
        "train": klue_sts_regression["train"],
        "test": klue_sts_regression["validation"],
    }
)

print(hf_dataset)

DatasetDict({
    train: Dataset({
        features: ['guid', 'source', 'sentence1', 'sentence2', 'labels', 'label', 'real_label', 'official_binary_label'],
        num_rows: 11668
    })
    test: Dataset({
        features: ['guid', 'source', 'sentence1', 'sentence2', 'labels', 'label', 'real_label', 'official_binary_label'],
        num_rows: 519
    })
})


`real-label` 점수 분포를 확인하고, positive가 약간 더 많이 나오도록 이진화 기준 threshold를 선택합니다. `TARGET_POSITIVE_RATIO`를 0.55로 두면 train 기준 positive 비율이 약 55%가 되도록 threshold를 자동으로 고릅니다.

In [4]:
import numpy as np

TARGET_POSITIVE_RATIO = 0.55
THRESHOLD_CANDIDATES = np.round(np.arange(0.0, 5.0001, 0.05), 2)


def choose_threshold_by_positive_ratio(scores, target_positive_ratio=0.55):
    scores = np.asarray(scores, dtype=float)
    rows = []
    for threshold in THRESHOLD_CANDIDATES:
        positive_ratio = float(np.mean(scores >= threshold))
        rows.append(
            {
                "threshold": float(threshold),
                "positive_ratio": positive_ratio,
                "distance": abs(positive_ratio - target_positive_ratio),
            }
        )
    return min(rows, key=lambda row: (row["distance"], -row["positive_ratio"]))


label_threshold_info = choose_threshold_by_positive_ratio(
    hf_dataset["train"]["real_label"],
    target_positive_ratio=TARGET_POSITIVE_RATIO,
)
LABEL_THRESHOLD = label_threshold_info["threshold"]

print("선택된 정답 이진화 threshold:", LABEL_THRESHOLD)
print("train positive 비율:", round(label_threshold_info["positive_ratio"], 4))

선택된 정답 이진화 threshold: 1.9
train positive 비율: 0.5495


선택한 threshold로 train/test의 positive/negative 비율을 다시 확인합니다. 이 비율은 공식 `binary-label`이 아니라 `real-label >= LABEL_THRESHOLD` 기준입니다.

In [5]:
def threshold_label_distribution(dataset, threshold):
    scores = np.asarray(dataset["real_label"], dtype=float)
    positives = int(np.sum(scores >= threshold))
    negatives = int(len(scores) - positives)
    return {
        "negative": {"count": negatives, "ratio": round(negatives / len(scores), 4)},
        "positive": {"count": positives, "ratio": round(positives / len(scores), 4)},
    }


for split_name, split_dataset in hf_dataset.items():
    print(split_name, len(split_dataset), threshold_label_distribution(split_dataset, LABEL_THRESHOLD))

train 11668 {'negative': {'count': 5256, 'ratio': 0.4505}, 'positive': {'count': 6412, 'ratio': 0.5495}}
test 519 {'negative': {'count': 202, 'ratio': 0.3892}, 'positive': {'count': 317, 'ratio': 0.6108}}


# 3. 토크나이저와 모델 설정

이번 버전에서는 모델을 `klue/roberta-base`로 변경합니다. 회귀 학습을 위해 `AutoModelForSequenceClassification`을 `num_labels=1`, `problem_type='regression'` 설정으로 사용합니다.

In [6]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL_NAME = "klue/roberta-base"

huggingface_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

정적 패딩을 적용하는 토크나이징 함수입니다. 동적 패딩과 bucketing 코드는 이번 버전에서 완전히 제외했습니다. 이전 설정과 동일하게 `MAX_LENGTH=128`을 사용합니다. `klue/roberta-base`는 `token_type_ids`를 사용하지 않으므로 CUDA 오류를 피하기 위해 토크나이저에서 해당 값을 만들지 않게 설정합니다.

In [7]:
MAX_LENGTH = 128


def transform(data):
    return huggingface_tokenizer(
        data["sentence1"],
        data["sentence2"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_token_type_ids=False,
    )

모든 split에 토크나이징을 적용합니다. 모델 학습에는 `label` 컬럼의 회귀값만 사용하고, threshold 분석용 `real_label`, `official_binary_label`은 토크나이징 전 데이터셋에 보존되어 있습니다.

In [8]:
columns_to_remove = [
    col for col in hf_dataset["train"].column_names if col != "label"
]

hf_tokenized_dataset = hf_dataset.map(
    transform,
    batched=True,
    remove_columns=columns_to_remove,
)

hf_train_dataset = hf_tokenized_dataset["train"]
hf_test_dataset = hf_tokenized_dataset["test"]

print(hf_train_dataset)
print(hf_train_dataset[0].keys())

Map:   0%|          | 0/11668 [00:00<?, ? examples/s]

Map:   0%|          | 0/519 [00:00<?, ? examples/s]

Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 11668
})
dict_keys(['label', 'input_ids', 'attention_mask'])


# 4. 회귀 평가 함수와 threshold 탐색 함수

모델은 유사도 점수를 예측하므로 MSE/MAE를 함께 계산합니다. accuracy, precision, recall, f1은 실제 점수와 예측 점수를 각각 threshold로 이진화한 뒤 계산합니다.

In [9]:
def binary_metrics_from_scores(true_scores, pred_scores, label_threshold, prediction_threshold):
    true_scores = np.asarray(true_scores, dtype=float)
    pred_scores = np.clip(np.asarray(pred_scores, dtype=float), 0.0, 5.0)

    true_binary = (true_scores >= label_threshold).astype(int)
    pred_binary = (pred_scores >= prediction_threshold).astype(int)

    accuracy = float(np.mean(pred_binary == true_binary))
    tp = int(np.sum((pred_binary == 1) & (true_binary == 1)))
    fp = int(np.sum((pred_binary == 1) & (true_binary == 0)))
    fn = int(np.sum((pred_binary == 0) & (true_binary == 1)))
    tn = int(np.sum((pred_binary == 0) & (true_binary == 0)))

    precision = float(tp / (tp + fp)) if (tp + fp) else 0.0
    recall = float(tp / (tp + fn)) if (tp + fn) else 0.0
    f1 = float(2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    pred_positive_ratio = float(np.mean(pred_binary == 1))
    true_positive_ratio = float(np.mean(true_binary == 1))

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
        "pred_positive_ratio": pred_positive_ratio,
        "true_positive_ratio": true_positive_ratio,
    }


PREDICTION_THRESHOLD = LABEL_THRESHOLD


def compute_regression_metrics(eval_pred):
    predictions, labels = eval_pred
    pred_scores = np.squeeze(predictions)
    true_scores = np.asarray(labels, dtype=float)
    clipped_pred_scores = np.clip(pred_scores, 0.0, 5.0)

    mse = float(np.mean((clipped_pred_scores - true_scores) ** 2))
    mae = float(np.mean(np.abs(clipped_pred_scores - true_scores)))
    binary_metrics = binary_metrics_from_scores(
        true_scores=true_scores,
        pred_scores=clipped_pred_scores,
        label_threshold=LABEL_THRESHOLD,
        prediction_threshold=PREDICTION_THRESHOLD,
    )

    return {
        "mse": mse,
        "mae": mae,
        "accuracy": binary_metrics["accuracy"],
        "precision": binary_metrics["precision"],
        "recall": binary_metrics["recall"],
        "f1": binary_metrics["f1"],
        "pred_positive_ratio": binary_metrics["pred_positive_ratio"],
    }

학습이 끝난 모델의 예측 점수를 대상으로 prediction threshold를 탐색합니다. accuracy를 우선으로 고르고, 동률이면 positive 비율이 목표값에 가까운 threshold를 선택합니다.

In [10]:
def find_best_prediction_threshold(pred_scores, true_scores, label_threshold, target_positive_ratio=0.55):
    rows = []
    for prediction_threshold in THRESHOLD_CANDIDATES:
        metrics = binary_metrics_from_scores(
            true_scores=true_scores,
            pred_scores=pred_scores,
            label_threshold=label_threshold,
            prediction_threshold=prediction_threshold,
        )
        rows.append(
            {
                "prediction_threshold": float(prediction_threshold),
                **metrics,
                "positive_ratio_distance": abs(metrics["pred_positive_ratio"] - target_positive_ratio),
            }
        )

    return max(
        rows,
        key=lambda row: (
            row["accuracy"],
            row["f1"],
            -row["positive_ratio_distance"],
        ),
    )

# 5. TrainingArguments 공통 설정

기존 설정을 최대한 유지하되 batch size는 16, epoch는 10으로 변경합니다. 현재 프로젝트에서 사용하던 `bf16`, `cosine` scheduler, 평가 batch size 64, dataloader worker 4 설정은 유지합니다. 회귀 모델의 best checkpoint는 threshold와 무관한 `mse` 기준으로 선택합니다.

In [ ]:
import gc
import os
import time
from pathlib import Path

import torch
import wandb
from transformers import Trainer, TrainingArguments

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16
optim_name = "adamw_torch_fused" if torch.cuda.is_available() else "adamw_torch"

print(f"bf16 사용여부: {use_bf16} / fp16 사용여부: {use_fp16}")
print(f"optimizer: {optim_name}")

COMMON_TRAINING_KWARGS = {
    "eval_strategy": "epoch",
    "save_strategy": "epoch",
    "per_device_train_batch_size": 16,
    "per_device_eval_batch_size": 64,
    "num_train_epochs": 10,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "lr_scheduler_type": "cosine",
    "optim": optim_name,
    "load_best_model_at_end": True,
    "metric_for_best_model": "mse",
    "greater_is_better": False,
    "save_total_limit": 2,
    "bf16": use_bf16,
    "fp16": use_fp16,
    "dataloader_num_workers": 4,
    "dataloader_pin_memory": True,
    "logging_steps": 50,
    "report_to": "wandb",
    "run_name": "2nd trial",
    "seed": 42,
}

LEARNING_RATES = [2e-5, 3e-5, 4e-5, 5e-5]
PROJECT_NAME = "HF_korean_project"
WANDB_RUN_NAME = "2nd trial"

bf16 사용여부: True / fp16 사용여부: False
optimizer: adamw_torch_fused


# 6. Learning rate 탐색 학습

`2e-5`, `3e-5`, `4e-5`, `5e-5` 네 가지 learning rate를 순서대로 학습합니다. `wandb.init`의 `name`은 요청대로 `2nd trial`로 고정하고, 각 run의 learning rate는 config로 구분합니다.

In [12]:
search_results = []

for learning_rate in LEARNING_RATES:
    lr_tag = f"{learning_rate:.0e}".replace("-", "m")
    output_dir = f"transformers_klue_sts_regression_roberta_lr_{lr_tag}"

    wandb.init(
        project=PROJECT_NAME,
        name=WANDB_RUN_NAME,
        reinit=True,
        config={
            "model_name": MODEL_NAME,
            "learning_rate": learning_rate,
            "batch_size": 16,
            "epochs": 10,
            "label_threshold": LABEL_THRESHOLD,
            "target_positive_ratio": TARGET_POSITIVE_RATIO,
            "padding": "max_length",
        },
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=1,
        problem_type="regression",
    )

    training_arguments = TrainingArguments(
        output_dir=output_dir,
        learning_rate=learning_rate,
        **COMMON_TRAINING_KWARGS,
    )

    trainer = Trainer(
        model=model,
        args=training_arguments,
        train_dataset=hf_train_dataset,
        eval_dataset=hf_test_dataset,
        compute_metrics=compute_regression_metrics,
    )

    start_time = time.time()
    train_result = trainer.train()
    train_seconds = time.time() - start_time

    prediction_output = trainer.predict(hf_test_dataset)
    pred_scores = np.squeeze(prediction_output.predictions)
    true_scores = np.asarray(prediction_output.label_ids, dtype=float)
    threshold_result = find_best_prediction_threshold(
        pred_scores=pred_scores,
        true_scores=true_scores,
        label_threshold=LABEL_THRESHOLD,
        target_positive_ratio=TARGET_POSITIVE_RATIO,
    )

    model_dir = Path(output_dir) / "best_model"
    trainer.save_model(model_dir)

    result = {
        "learning_rate": learning_rate,
        "output_dir": output_dir,
        "model_dir": str(model_dir),
        "train_seconds": round(train_seconds, 2),
        "train_loss": train_result.metrics.get("train_loss"),
        **threshold_result,
    }
    search_results.append(result)

    wandb.log(
        {
            "best_prediction_threshold": result["prediction_threshold"],
            "threshold_tuned_accuracy": result["accuracy"],
            "threshold_tuned_f1": result["f1"],
            "threshold_tuned_precision": result["precision"],
            "threshold_tuned_recall": result["recall"],
            "threshold_tuned_pred_positive_ratio": result["pred_positive_ratio"],
            "train_seconds": result["train_seconds"],
        }
    )
    wandb.finish()

    del trainer
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("learning rate 탐색 완료")

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\Sungwoo\_netrc.
wandb: Currently logged in as: ohmanbo (ohmanbo-modulab) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: klue/roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Mse,Mae,Accuracy,Precision,Recall,F1,Pred Positive Ratio
1,0.243736,0.470739,0.470739,0.522270,0.909441,0.899408,0.958991,0.928244,0.651252
2,0.164101,0.427102,0.427102,0.479341,0.909441,0.881356,0.984227,0.929955,0.682081
3,0.120071,0.365151,0.365150,0.449259,0.907514,0.874652,0.990536,0.928994,0.691715
4,0.095413,0.387601,0.387330,0.476092,0.903661,0.865753,0.996845,0.926686,0.703276
5,0.059907,0.335430,0.335131,0.420432,0.919075,0.891738,0.987382,0.937126,0.676301
6,0.053398,0.426431,0.426431,0.477245,0.882466,0.842246,0.993691,0.911722,0.720617
7,0.034937,0.336666,0.336665,0.419938,0.924855,0.897143,0.990536,0.941529,0.674374
8,0.027665,0.389468,0.389410,0.456015,0.903661,0.867769,0.993691,0.926471,0.699422
9,0.025018,0.359534,0.359515,0.438621,0.913295,0.879888,0.993691,0.933333,0.689788
10,0.022665,0.357630,0.357571,0.437193,0.911368,0.879552,0.990536,0.931751,0.687861


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

best_prediction_threshold,▁
eval/accuracy,▇▄▆▁██▇▆▆▆
eval/f1,▇▄▆▁██▇▆▆▆
eval/loss,▄█▃▆▁▁▁▂▁▁
eval/mae,▄▇▃█▂▁▁▂▁▁
eval/mse,▄█▃▆▁▁▁▂▁▁
eval/precision,█▃▅▁▇▇▆▅▅▅
eval/pred_positive_ratio,▁▆▅█▃▃▄▅▅▅
eval/recall,▁█▇█▆▆▆▇▇▇
eval/runtime,▁▃▂▂▂▂█▄▄▄
+24,...


learning rate 탐색 완료


네 가지 learning rate 결과를 비교하고, accuracy가 가장 높은 설정을 선택합니다. 동률일 경우 f1이 더 높은 결과를 우선합니다.

In [13]:
for result in search_results:
    print(
        f"lr={result['learning_rate']:.0e}",
        f"accuracy={result['accuracy']:.4f}",
        f"f1={result['f1']:.4f}",
        f"threshold={result['prediction_threshold']:.2f}",
        f"pred_positive_ratio={result['pred_positive_ratio']:.4f}",
        f"time={result['train_seconds']}s",
    )

best_result = max(search_results, key=lambda row: (row["accuracy"], row["f1"]))
BEST_LEARNING_RATE = best_result["learning_rate"]
PREDICTION_THRESHOLD = best_result["prediction_threshold"]

print()
print("선택된 learning rate:", BEST_LEARNING_RATE)
print("선택된 prediction threshold:", PREDICTION_THRESHOLD)
print("선택된 모델 경로:", best_result["model_dir"])

lr=2e-05 accuracy=0.9383 f1=0.9498 threshold=2.25 pred_positive_ratio=0.6185 time=708.07s
lr=3e-05 accuracy=0.9461 f1=0.9560 threshold=2.35 pred_positive_ratio=0.6146 time=788.65s
lr=4e-05 accuracy=0.9383 f1=0.9511 threshold=2.15 pred_positive_ratio=0.6493 time=540.65s
lr=5e-05 accuracy=0.9345 f1=0.9459 threshold=2.45 pred_positive_ratio=0.5992 time=557.46s

선택된 learning rate: 3e-05
선택된 prediction threshold: 2.35
선택된 모델 경로: transformers_klue_sts_regression_roberta_lr_3em05\best_model


# 7. 선택된 모델 최종 평가

가장 좋은 learning rate로 저장된 모델을 다시 불러오고, 선택된 prediction threshold를 적용해 최종 평가 지표를 출력합니다.

In [14]:
best_model = AutoModelForSequenceClassification.from_pretrained(best_result["model_dir"])

final_eval_args = TrainingArguments(
    output_dir="transformers_klue_sts_regression_roberta_best_eval",
    per_device_eval_batch_size=64,
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    report_to="none",
)

final_trainer = Trainer(
    model=best_model,
    args=final_eval_args,
    eval_dataset=hf_test_dataset,
    compute_metrics=compute_regression_metrics,
)

final_eval_results = final_trainer.evaluate(hf_test_dataset)
print(final_eval_results)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Training Loss,Validation Loss,Step,Mse,Mae,Accuracy,Precision,Recall,F1,Pred Positive Ratio
No log,0.332859,0,0.332799,0.428261,0.946050,0.952978,0.958991,0.955975,0.614644


{'eval_loss': 0.33285850286483765, 'eval_mse': 0.3327992324688497, 'eval_mae': 0.4282605896357569, 'eval_accuracy': 0.9460500963391136, 'eval_precision': 0.9529780564263323, 'eval_recall': 0.9589905362776026, 'eval_f1': 0.9559748427672956, 'eval_pred_positive_ratio': 0.6146435452793835}


최종 모델의 confusion matrix와 positive 비율을 출력합니다. 여기서의 positive/negative는 `real-label` 기반 threshold로 만든 라벨입니다.

In [15]:
final_prediction_output = final_trainer.predict(hf_test_dataset)
final_pred_scores = np.squeeze(final_prediction_output.predictions)
final_true_scores = np.asarray(final_prediction_output.label_ids, dtype=float)

final_binary_metrics = binary_metrics_from_scores(
    true_scores=final_true_scores,
    pred_scores=final_pred_scores,
    label_threshold=LABEL_THRESHOLD,
    prediction_threshold=PREDICTION_THRESHOLD,
)

print("[최종 평가 지표]")
print("best learning rate:", BEST_LEARNING_RATE)
print("label threshold:", LABEL_THRESHOLD)
print("prediction threshold:", PREDICTION_THRESHOLD)
print(final_binary_metrics)

[최종 평가 지표]
best learning rate: 3e-05
label threshold: 1.9
prediction threshold: 2.35
{'accuracy': 0.9460500963391136, 'precision': 0.9529780564263323, 'recall': 0.9589905362776026, 'f1': 0.9559748427672956, 'tp': 304, 'fp': 15, 'fn': 13, 'tn': 187, 'pred_positive_ratio': 0.6146435452793835, 'true_positive_ratio': 0.6107899807321773}


마지막으로 메모리를 정리합니다. 다음 실험을 이어서 실행할 때 GPU 캐시가 남아 생기는 문제를 줄이기 위한 셀입니다.

In [16]:
del best_model
del final_trainer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("메모리 정리 완료")

메모리 정리 완료


# 8. 직접 작성한 예문 테스트 함수

학습과 최종 평가가 끝난 뒤, 직접 작성한 두 문장을 넣어 유사도 점수와 positive/negative 판정을 확인하는 함수입니다. 마지막 메모리 정리 셀을 실행해 `best_model`이 삭제된 상태여도 `best_result["model_dir"]`에서 모델을 다시 불러오도록 만들었습니다.

In [18]:
def predict_sts_pair(sentence1, sentence2, model_dir=None, prediction_threshold=None, max_length=MAX_LENGTH):
    """직접 작성한 한국어 문장쌍의 STS 유사도와 positive/negative 판정을 반환합니다."""
    import numpy as np
    import torch
    from transformers import AutoModelForSequenceClassification

    if prediction_threshold is None:
        prediction_threshold = PREDICTION_THRESHOLD

    if model_dir is None:
        model_dir = best_result["model_dir"]

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    inference_model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(device)
    inference_model.eval()

    encoded = huggingface_tokenizer(
        sentence1,
        sentence2,
        truncation=True,
        padding="max_length",
        max_length=max_length,
        return_token_type_ids=False,
        return_tensors="pt",
    )
    encoded = {key: value.to(device) for key, value in encoded.items()}

    with torch.no_grad():
        output = inference_model(**encoded)

    predicted_score = float(np.clip(output.logits.squeeze().detach().cpu().item(), 0.0, 5.0))
    predicted_label = "positive" if predicted_score >= prediction_threshold else "negative"

    result = {
        "sentence1": sentence1,
        "sentence2": sentence2,
        "predicted_real_label": round(predicted_score, 4),
        "prediction_threshold": prediction_threshold,
        "predicted_label": predicted_label,
    }

    del inference_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result


def predict_sts_examples(sentence_pairs, model_dir=None, prediction_threshold=None):
    """[(문장1, 문장2), ...] 형태의 여러 문장쌍을 한 번에 테스트합니다."""
    results = []
    for sentence1, sentence2 in sentence_pairs:
        results.append(
            predict_sts_pair(
                sentence1,
                sentence2,
                model_dir=model_dir,
                prediction_threshold=prediction_threshold,
            )
        )
    return results


# 사용 예시: 문장쌍 2개를 원하는 문장으로 바꿔서 실행하세요.
my_examples = [
    ("오늘 날씨가 정말 좋다.", "오늘은 날씨가 참 맑다."),
    ("나는 점심으로 김치찌개를 먹었다.", "그 영화는 결말이 매우 슬펐다."),
]

predict_sts_examples(my_examples)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[{'sentence1': '오늘 날씨가 정말 좋다.',
  'sentence2': '오늘은 날씨가 참 맑다.',
  'predicted_real_label': 4.5334,
  'prediction_threshold': 2.35,
  'predicted_label': 'positive'},
 {'sentence1': '나는 점심으로 김치찌개를 먹었다.',
  'sentence2': '그 영화는 결말이 매우 슬펐다.',
  'predicted_real_label': 0.0,
  'prediction_threshold': 2.35,
  'predicted_label': 'negative'}]

In [19]:
sentence_1 = "황두용은 AIFFEL에서 딥러닝을 공부하고 있다."
sentence_2 = "황씨 성을가진사람이 AIFFEL에 있는데 딥러닝을 공부하고있고 이름이 두용이래."
predict_sts_pair(sentence_1, sentence_2)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

{'sentence1': '황두용은 AIFFEL에서 딥러닝을 공부하고 있다.',
 'sentence2': '황씨 성을가진사람이 AIFFEL에 있는데 딥러닝을 공부하고있고 이름이 두용이래.',
 'predicted_real_label': 3.5716,
 'prediction_threshold': 2.35,
 'predicted_label': 'positive'}

In [20]:
sentence_1 = "황두용은 AIFFEL에서 딥러닝을 공부하고 있다."
sentence_2 = "황씨 성을가진사람이 AIFFEL에 있는데 딥러닝을 공부하고있고 이름이 삼용이래."
predict_sts_pair(sentence_1, sentence_2)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

{'sentence1': '황두용은 AIFFEL에서 딥러닝을 공부하고 있다.',
 'sentence2': '황씨 성을가진사람이 AIFFEL에 있는데 딥러닝을 공부하고있고 이름이 삼용이래.',
 'predicted_real_label': 3.3377,
 'prediction_threshold': 2.35,
 'predicted_label': 'positive'}

In [21]:
sentence_1 = "황두용은 AIFFEL에서 딥러닝을 공부하고 있다."
sentence_2 = "황씨 성을가진사람이 AIFFEL에 있는데 게임개발을 공부하고있고 이름이 두용이래."
predict_sts_pair(sentence_1, sentence_2)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

{'sentence1': '황두용은 AIFFEL에서 딥러닝을 공부하고 있다.',
 'sentence2': '황씨 성을가진사람이 AIFFEL에 있는데 게임개발을 공부하고있고 이름이 두용이래.',
 'predicted_real_label': 2.6038,
 'prediction_threshold': 2.35,
 'predicted_label': 'positive'}

In [22]:
sentence_1 = "황두용은 AIFFEL에서 딥러닝을 공부하고 있다."
sentence_2 = "김씨 성을가진사람이 AIFFEL에 있는데 딥러닝을 공부하고있고 이름이 두용이래."
predict_sts_pair(sentence_1, sentence_2)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

{'sentence1': '황두용은 AIFFEL에서 딥러닝을 공부하고 있다.',
 'sentence2': '김씨 성을가진사람이 AIFFEL에 있는데 딥러닝을 공부하고있고 이름이 두용이래.',
 'predicted_real_label': 2.9727,
 'prediction_threshold': 2.35,
 'predicted_label': 'positive'}

In [23]:
sentence_1 = "황두용은 AIFFEL에서 딥러닝을 공부하고 있다."
sentence_2 = "김씨 성을가진사람이 APPLE에 있는데 맥북을 공부하고있고 이름이 두용이래."
predict_sts_pair(sentence_1, sentence_2)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

{'sentence1': '황두용은 AIFFEL에서 딥러닝을 공부하고 있다.',
 'sentence2': '김씨 성을가진사람이 APPLE에 있는데 맥북을 공부하고있고 이름이 두용이래.',
 'predicted_real_label': 0.1364,
 'prediction_threshold': 2.35,
 'predicted_label': 'negative'}

In [24]:
sentence_1 = "ㅋㅋㅋ"
sentence_2 = "ㅎㅎㅎ"
predict_sts_pair(sentence_1, sentence_2)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

{'sentence1': 'ㅋㅋㅋ',
 'sentence2': 'ㅎㅎㅎ',
 'predicted_real_label': 2.32,
 'prediction_threshold': 2.35,
 'predicted_label': 'negative'}

In [25]:
sentence_1 = "ㅋㅋ"
sentence_2 = "ㅜㅜ"
predict_sts_pair(sentence_1, sentence_2)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

{'sentence1': 'ㅋㅋ',
 'sentence2': 'ㅜㅜ',
 'predicted_real_label': 0.9027,
 'prediction_threshold': 2.35,
 'predicted_label': 'negative'}

In [26]:
sentence_1 = "ㅠㅠ"
sentence_2 = "ㅜㅜ"
predict_sts_pair(sentence_1, sentence_2)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

{'sentence1': 'ㅠㅠ',
 'sentence2': 'ㅜㅜ',
 'predicted_real_label': 1.4529,
 'prediction_threshold': 2.35,
 'predicted_label': 'negative'}

In [27]:
sentence_1 = "맥북이 있으면 스타벅스에 입장할수 있다."
sentence_2 = "노트북이 있으면 카페에 입장할수 있다."
predict_sts_pair(sentence_1, sentence_2)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

{'sentence1': '맥북이 있으면 스타벅스에 입장할수 있다.',
 'sentence2': '노트북이 있으면 카페에 입장할수 있다.',
 'predicted_real_label': 2.638,
 'prediction_threshold': 2.35,
 'predicted_label': 'positive'}

In [28]:
sentence_1 = "맥북이 있으면 스타벅스에 입장할수 있다."
sentence_2 = "비싼시계가 있으면 카페에 입장할수 있다."
predict_sts_pair(sentence_1, sentence_2)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

{'sentence1': '맥북이 있으면 스타벅스에 입장할수 있다.',
 'sentence2': '비싼시계가 있으면 카페에 입장할수 있다.',
 'predicted_real_label': 1.3448,
 'prediction_threshold': 2.35,
 'predicted_label': 'negative'}

In [29]:
sentence_1 = "이종현님은 매일 매일 바쁜 하루를 보낸다."
sentence_2 = "AIFFEL에서 가장 친절한 사람은 이종현님이다."
predict_sts_pair(sentence_1, sentence_2)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

{'sentence1': '이종현님은 매일 매일 바쁜 하루를 보낸다.',
 'sentence2': 'AIFFEL에서 가장 친절한 사람은 이종현님이다.',
 'predicted_real_label': 1.18,
 'prediction_threshold': 2.35,
 'predicted_label': 'negative'}

In [30]:
sentence_1 = "이종현님은 매일 매일 바쁜 하루를 보낸다."
sentence_2 = "AIFFEL에서 가장 바쁜사람은 이종현님이다."
predict_sts_pair(sentence_1, sentence_2)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

{'sentence1': '이종현님은 매일 매일 바쁜 하루를 보낸다.',
 'sentence2': 'AIFFEL에서 가장 바쁜사람은 이종현님이다.',
 'predicted_real_label': 2.4238,
 'prediction_threshold': 2.35,
 'predicted_label': 'positive'}